##### What this notebook for:

It interacts strictly with external cloud resources and the AWS infrastructure layer.

##### Its Core Job:

It takes the secret download URL, pulls the heavy .zip archive straight into the instance's temporary memory buffer (/tmp/), extracts the raw .tsv files, uploads them safely to your Amazon S3 bucket, and then self-destructs the temporary folder completely to keep your storage bill at zero.

##### Frequency:

We only run this notebook once at the very beginning of the hackathon to safely move the data from Unstop into your cloud playground. Once your S3 bucket is populated, you never need to run it again.

In [ ]:
import os

# Define the canonical relative pathways
base_src_path = "AWS_ML_challengees-2026Launch-01s_/Package/code/business_entity_resolution/src"

# Create directories for source code files, pre-compiled wheels, and models
os.makedirs(os.path.join(base_src_path, "models"), exist_ok=True)
os.makedirs(os.path.join(base_src_path, "wheels"), exist_ok=True)

print("-> Root workspace folders successfully staged inside SageMaker Studio.")

(Note: S3 bucket names must be entirely unique globally).

### Run this command to create a unique bucket in your active AWS region:

_____________________________________________________________________________

#### bash
===============================================================================================

    aws s3api create-bucket \
      --bucket amazon-ml-challenge-2026-team-unique-id \
      --region ap-north-1 \
      --create-bucket-configuration LocationConstraint=ap-north-1
  
  ---------------------------------------------------------------------------

In [ ]:
import os
import zipfile
import urllib.request
import shutil
import boto3

# Dataset link generated for challenge workspace
challenge_data_url = '' # You need to fill it so that notebook can access data stream. It could be a link that would look like this - 'https://fg4syt3j.r.ap-north-1.awstrack.me/L0/https:%7F%7Fcdn.unstop.com%..'. We already had processed data, we have checked that the notebook architecture is working, but the dataset we processed was extremely large. So we wouldn't be providing any data in any file of the repository, the data-related files will be blank.

# Isolated scratchpad directories
tmp_dir = "/tmp/challenge_data_staging"
zip_path = os.path.join(tmp_dir, "student_resource.zip")
extract_path = os.path.join(tmp_dir, "extracted")

os.makedirs(tmp_dir, exist_ok=True)
s3_bucket = "amazon-ml-challenge-2026-simple-storage-bucket"  # <── Change this to your bucket name
s3_client = boto3.client('s3')

print("-> Streaming challenge zip directly into instance /tmp/ partition...")
urllib.request.urlretrieve(unstop_url, zip_path)

print("-> Extracting challenge folder layers...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("-> Uploading unzipped .tsv files to Amazon S3...")
for root, dirs, files in os.walk(extract_path):
    for file in files:
        if file.endswith('.tsv'):
            local_file_path = os.path.join(root, file)
            # Reformat file path paths for S3 keys
            relative_prefix = os.path.relpath(local_file_path, extract_path).replace("\\", "/")
            s3_key = f"raw-data/{relative_prefix}"
            
            print(f"   Uploading: {file} ──► s3://{s3_bucket}/{s3_key}")
            s3_client.upload_file(local_file_path, s3_bucket, s3_key)

print("-> Initiating temporary pipeline wipe...")
shutil.rmtree(tmp_dir)
print("-> Volatile staging environment successfully cleared! Zero local persistent disk used.")


Concluding what we have done above here.

```

[Download Link] 
         │ 
         ▼ (Temporary Volatile Stream)
┌────────────────────────────────┐
│   Instance /tmp/ Scratchpad    │ <── (Wiped out instantly post-run)
└────────────────────────────────┘
         │
         ▼ (Programmatic Push via Boto3)
┌────────────────────────────────┐
│   Amazon S3 Bucket Storage     │ <── Data lives here permanently
└────────────────────────────────┘
         │
         ▼ (Stream on-demand via Fast File Mode)
┌────────────────────────────────┐
│ SageMaker JupyterLab Space     │ <── Notebook disk stays empty
└────────────────────────────────┘

```